
# FX Data Loading into PostgreSQL

## Phase 5 — Database Integration

### Business Objective
Load validated USD/INR, EUR/INR, and GBP/INR exchange-rate observations into PostgreSQL.

### Source Dataset
`Data/processed/fx_rates_processed.csv`

### Target Database
`financial_market_analytics`

### Target Table
`public.fx_rates`

### Expected Records
312 observations:
- USD/INR: 104
- EUR/INR: 104
- GBP/INR: 104

### Loading Strategy
1. Read the validated CSV dataset.
2. Establish a secure PostgreSQL connection using environment variables.
3. Insert records into the existing database table.
4. Preserve primary keys and database constraints.
5. Validate row counts and currency distributions.

### Data Integrity
Do not recreate or replace the existing PostgreSQL table.
Do not overwrite the raw or processed CSV datasets.


In [1]:

from pathlib import Path
import sys

# Locate the project root.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("FX_PostgreSQL_Loading")

logger.info("FX PostgreSQL loading notebook initialized")


2026-11-02 16:08:58,569 | INFO | FX_PostgreSQL_Loading | FX PostgreSQL loading notebook initialized


In [1]:

from pathlib import Path
import pandas as pd

# Locate project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Define validated FX dataset path
FX_FILE = (
    PROJECT_ROOT / "Data" / "processed"
    / "fx_rates_processed.csv"
)

# Verify file exists
assert FX_FILE.exists(), (
    f"Processed FX dataset not found: {FX_FILE}"
)

# Load processed dataset
df_fx_load = pd.read_csv(
    FX_FILE,
    parse_dates=["observation_date"]
)

# Inspect dataset
print("Dataset Shape:", df_fx_load.shape)

print("\nColumn Names:")
print(df_fx_load.columns.tolist())

print("\nCurrency Distribution:")
print(
    df_fx_load.groupby(
        ["base_currency", "quote_currency"]
    ).size()
)

print("\nMissing Values:")
print(df_fx_load.isna().sum())

print("\nDataset Preview:")
display(df_fx_load.head())

# Initial validation
assert len(df_fx_load) == 312
assert df_fx_load.isna().sum().sum() == 0

print("\nFX Dataset Pre-Loading Validation: PASSED")


Dataset Shape: (312, 6)

Column Names:
['observation_date', 'base_currency', 'quote_currency', 'exchange_rate', 'source', 'is_outlier']

Currency Distribution:
base_currency  quote_currency
EUR            INR               104
GBP            INR               104
USD            INR               104
dtype: int64

Missing Values:
observation_date    0
base_currency       0
quote_currency      0
exchange_rate       0
source              0
is_outlier          0
dtype: int64

Dataset Preview:


,observation_date,base_currency,quote_currency,exchange_rate,source,is_outlier
0,2026-05-21,EUR,INR,111.5885,Frankfurter,False
1,2026-05-22,EUR,INR,110.9585,Frankfurter,False
2,2026-05-25,EUR,INR,110.8870,Frankfurter,False
3,2026-05-26,EUR,INR,111.3200,Frankfurter,False
4,2026-05-27,EUR,INR,111.3650,Frankfurter,False



FX Dataset Pre-Loading Validation: PASSED



## 2. Secure PostgreSQL Database Connection

### Business Objective
Establish a secure connection between Python and the PostgreSQL database.

### Database Configuration
- Database: financial_market_analytics
- Database system: PostgreSQL
- Connection library: SQLAlchemy
- Credentials source: .env

### Validation Method
1. Load environment variables securely.
2. Create the PostgreSQL connection engine.
3. Connect to the database.
4. Verify the connected database name.
5. Confirm that the FX target table exists.
6. Check the existing record count.

### Security
Database passwords must not be hardcoded, printed, or uploaded to GitHub.

In [6]:

import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Locate environment configuration
ENV_PATH = PROJECT_ROOT / ".env"

assert ENV_PATH.exists(), ".env file not found."

# Load database credentials
load_dotenv(ENV_PATH)

required_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

missing_variables = [
    key for key in required_variables
    if not os.getenv(key)
]

assert not missing_variables, (
    f"Missing environment variables: {missing_variables}"
)

# Build connection URL securely
database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

# Create SQLAlchemy engine
engine = create_engine(
    database_url,
    pool_pre_ping=True
)

# Test database connection
with engine.connect() as connection:

    database_name = connection.execute(
        text("SELECT current_database()")
    ).scalar_one()

    table_exists = connection.execute(
        text("""
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_schema = 'public'
                  AND table_name = 'fx_rates'
            )
        """)
    ).scalar_one()

    existing_records = connection.execute(
        text("SELECT COUNT(*) FROM public.fx_rates")
    ).scalar_one()

print("PostgreSQL Connection: SUCCESS")
print("Connected Database:", database_name)
print("FX Table Exists:", table_exists)
print("Existing FX Records:", existing_records)

assert database_name == "financial_market_analytics"
assert table_exists

print("\nPostgreSQL Connection Validation: PASSED")


PostgreSQL Connection: SUCCESS
Connected Database: financial_market_analytics
FX Table Exists: True
Existing FX Records: 312

PostgreSQL Connection Validation: PASSED


In [5]:

# Reuse the project root configured during logger initialization.
PROJECT_ROOT = project_root

print("Project Root:", PROJECT_ROOT)
print(".env File Exists:", (PROJECT_ROOT / ".env").is_file())


Project Root: E:\BA Projects\Financial_Market_Intelligence_Analytics
.env File Exists: True


In [7]:

from sqlalchemy import text

# Read-only verification of existing PostgreSQL data.
with engine.connect() as connection:
    db_fx_count = connection.execute(
        text("SELECT COUNT(*) FROM public.fx_rates")
    ).scalar_one()

# Record the verification results.
logger.info("FX PostgreSQL record verification started")
logger.info("Database table: public.fx_rates")
logger.info("Existing FX records: %s", db_fx_count)

if db_fx_count == 312:
    logger.info("FX PostgreSQL record count validation PASSED")
else:
    logger.error(
        "FX PostgreSQL record count validation FAILED | "
        "Expected: 312 | Actual: %s",
        db_fx_count
    )


2026-11-02 16:15:41,515 | INFO | FX_PostgreSQL_Loading | FX PostgreSQL record verification started
2026-11-02 16:15:42,026 | INFO | FX_PostgreSQL_Loading | Database table: public.fx_rates
2026-11-02 16:15:42,033 | INFO | FX_PostgreSQL_Loading | Existing FX records: 312
2026-11-02 16:15:42,055 | INFO | FX_PostgreSQL_Loading | FX PostgreSQL record count validation PASSED



## 3. Load Validated FX Data into PostgreSQL

### Business Objective
Insert validated historical exchange-rate observations into the PostgreSQL fx_rates table.

### Source
Data/processed/fx_rates_processed.csv

### Target
financial_market_analytics.public.fx_rates

### Expected Records
- USD/INR: 104
- EUR/INR: 104
- GBP/INR: 104
- Total: 312

### Loading Method
1. Verify the source dataset.
2. Confirm that the target table is empty.
3. Insert the validated records into the existing table.
4. Preserve PostgreSQL primary keys and constraints.
5. Verify the inserted record count.

### Data Integrity
- Do not replace or recreate the database table.
- Do not modify the source CSV.
- Use a transaction to prevent partial loading.
- Prevent accidental duplicate insertion.

In [3]:

from sqlalchemy import text

# Verify source dataset before insertion
assert len(df_fx_load) == 312

assert df_fx_load.isna().sum().sum() == 0

assert not df_fx_load.duplicated(
    subset=[
        "observation_date",
        "base_currency",
        "quote_currency"
    ]
).any()

# Ensure the target table is empty
with engine.connect() as connection:
    existing_count = connection.execute(
        text("SELECT COUNT(*) FROM public.fx_rates")
    ).scalar_one()

assert existing_count == 0, (
    f"Target table contains {existing_count} records. "
    "Loading stopped to prevent duplicate insertion."
)

# Insert records within a transaction
with engine.begin() as connection:

    inserted_rows = df_fx_load.to_sql(
        name="fx_rates",
        con=connection,
        schema="public",
        if_exists="append",
        index=False,
        chunksize=100,
        method="multi"
    )

print("FX Data Insertion: COMPLETED")
print("Rows Reported Inserted:", inserted_rows)

# Verify database record count
with engine.connect() as connection:

    loaded_count = connection.execute(
        text("SELECT COUNT(*) FROM public.fx_rates")
    ).scalar_one()

assert loaded_count == len(df_fx_load), (
    "Database row count does not match source dataset."
)

print("Source Records:", len(df_fx_load))
print("Database Records:", loaded_count)
print("Record Count Match:", loaded_count == len(df_fx_load))

print("\nFX PostgreSQL Data Loading: PASSED")


FX Data Insertion: COMPLETED
Rows Reported Inserted: 312
Source Records: 312
Database Records: 312
Record Count Match: True

FX PostgreSQL Data Loading: PASSED
